# StopWordsRemover

Stop words are words which should be excluded from the input, typically because the words appear frequently and don’t carry as much meaning.

StopWordsRemover takes as input a sequence of strings (e.g. the output of a Tokenizer) and drops all the stop words from the input sequences. The list of stopwords is specified by the stopWords parameter. Default stop words for some languages are accessible by calling StopWordsRemover.loadDefaultStopWords(language), for which available options are “danish”, “dutch”, “english”, “finnish”, “french”, “german”, “hungarian”, “italian”, “norwegian”, “portuguese”, “russian”, “spanish”, “swedish” and “turkish”. A boolean parameter caseSensitive indicates if the matches should be case sensitive (false by default).

## Create PySpark environment and import library

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.master('local').appName('MyStopwords').getOrCreate()

from pyspark.ml.feature import Tokenizer, RegexTokenizer,StopWordsRemover
from pyspark.sql.functions import col, udf
from pyspark.sql.types import IntegerType

# Stopwords removal

In [2]:
sentenceDataFrame = spark.createDataFrame([
    (0, ['I', 'saw', 'the', 'green', 'horse']),
    (1, ['Mary', 'had', 'a', 'little', 'lamb'])
], ['id', 'tokens'])

sentenceDataFrame.show(truncate=False)

# In filtered, the stop words “I”, “the”, and “a” have been filtered out.
#remover = StopWordsRemover(inputCol = 'tokens', outputCol = 'filtered')

#you can also provide your selected stop words
mystopWords=['the', 'a', 'I']

# provide a specified stop words list: mystopWords
remover = StopWordsRemover(inputCol = 'tokens', outputCol = 'filtered',stopWords=mystopWords, caseSensitive=False)

filtered = remover.transform(sentenceDataFrame)
filtered.select(['filtered']).show(truncate=False)

+---+----------------------------+
|id |tokens                      |
+---+----------------------------+
|0  |[I, saw, the, green, horse] |
|1  |[Mary, had, a, little, lamb]|
+---+----------------------------+

+-------------------------+
|filtered                 |
+-------------------------+
|[saw, green, horse]      |
|[Mary, had, little, lamb]|
+-------------------------+



## Process text file input

First tokenize, then remove stop words

In [3]:
df_text = spark.read.text("data/2.txt")

df_text.show()

tokenizer = Tokenizer(inputCol = 'value', outputCol = 'words')
tokenized = tokenizer.transform(df_text)

#using default stop words
remover = StopWordsRemover(inputCol = 'words', outputCol = 'filtered')
#remover.transform(tokenized).show(truncate=False)

filtered = remover.transform(tokenized).select(['filtered']).show(truncate=False)

+--------------------+
|               value|
+--------------------+
|Lorem Ipsum is si...|
|The standard chun...|
|There are many va...|
|Lorem ipsum dolor...|
|Mauris ut dapibus...|
|Lorem Ipsum is si...|
|Contrary to popul...|
|The standard chun...|
+--------------------+

+---------------------------------------------------------------------------------------------+
|filtered                                                                                     |
+---------------------------------------------------------------------------------------------+
|[lorem, ipsum, simply, dummy, text, printing]                                                |
|[standard, chunk, lorem, ipsum, used, since, 1500s]                                          |
|[many, variations, passages, lorem, ipsum, available,, majority, suffered, alteration, form.]|
|[lorem, ipsum, dolor, sit, amet,, consectetur, adipiscing, elit.]                            |
|[mauris, ut, dapibus, lacus., maecenas, sollicitud

In [4]:
df_text = spark.read.text("data/1.txt")

#df_text.show(truncate=False)
df_text.show()


# you are only interested in English word, no other characters, i.e. numbers, or punctuations 
# you can use pattern="[a-zA-Z]+"
tokenizer = RegexTokenizer(inputCol = 'value', outputCol = 'words', minTokenLength=3,
                           gaps=False,
                           pattern="[a-zA-Z]+") 
tokenized = tokenizer.transform(df_text)

remover = StopWordsRemover(inputCol = 'words', outputCol = 'filtered')
#filtered = remover.transform(tokenized).show(truncate=False)
filtered = remover.transform(tokenized).select(['filtered']).show(truncate=False)

+--------------------+
|               value|
+--------------------+
|0001 Lorem Ipsum ...|
|0002 Contrary to ...|
|0003 The standard...|
|0004 It is a long...|
|0005 There are ma...|
|0006 Lorem ipsum ...|
|0007 Mauris ut da...|
|0008 Lorem Ipsum ...|
|0009 Contrary to ...|
|0010 The standard...|
|0011 It is a long...|
|0012 There are ma...|
+--------------------+

+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [6]:
spark.stop()